# Session 2: Univariate Analysis — Transactions and Products

## What This Notebook Does

Grouping and aggregations told us what happened at a group level.
Univariate analysis goes one level deeper — it examines each individual
column in isolation to understand its shape, spread, and outliers.
The files used are `transactions_clean.csv` and `products.csv`.

---

## Questions Covered

**Units Sold**
- What does the distribution of daily units sold look like across the year?
- Which products have the most unpredictable demand day to day?
- Which product produces the most extreme outlier sales days and what do those spikes represent?

**Revenue**
- How volatile is daily revenue relative to its average?
- Does revenue skew more than units sold — and why?
- Which product dominates the revenue outliers and is it the same product that dominates sales outliers?

**Gross Profit**
- How does gross profit volatility compare to revenue volatility?
- What does the difference between the two tell us about business health?

**Products Reference Table**
- How wide is the price range across the five products and what does that mean for grouped analysis?
- Which products share similar margins but differ in absolute profit per unit?
- Does the base demand setting in the reference table reflect actual sales behaviour in the transaction data?

---

## Files Used
- `transactions_clean.csv` — daily sales, revenue, cogs, and gross profit per product
- `products.csv` — product reference table with unit cost, unit price, category, and base demand

In [1]:
# importing essential libraries
import numpy as np
import pandas as pd

In [2]:
# loading the dataset
transactions_data = pd.read_csv('../../data/transactions_clean.csv')
products_data = pd.read_csv('../../data/products.csv')

In [3]:
transactions_data.head(5)

,date,product_id,units_sold,revenue,cogs,gross_profit
0,2024-01-01,P001,15,1349.85,675.0,674.85
1,2024-01-01,P002,16,479.84,192.0,287.84
2,2024-01-01,P003,17,339.83,136.0,203.83
3,2024-01-01,P004,12,719.88,360.0,359.88
4,2024-01-01,P005,5,649.95,275.0,374.95


In [4]:
transactions_data['units_sold'].describe().loc["50%"]

np.float64(21.0)

In [5]:
def units_sold_summary(df):
    """Showing all the statistical information of units sold column"""
    
    print("Statistical summary of units sold")
    display(df['units_sold'].describe())

def mean_median_comparison(df):
    """Comparing the mean and median of units sold column to guess the distribution"""
    mean = round(df['units_sold'].mean(), 2)
    median = round(df['units_sold'].median(), 2)
    difference = round(abs(mean - median), 2)
    
    if mean > median:
        print(f"The mean value ({mean}) is higher than the median ({median}) by {difference}.")
        print(f"This indicates a right-skewed distribution pulled toward high-sales days.")
    elif median > mean:
        print(f"The median value ({median}) is higher than the mean ({mean}) by {difference}.")
        print(f"This indicates a left-skewed distribution pulled toward low-sales days.")
    else:
        print("The mean and median are equal, indicating a perfectly balanced, symmetrical distribution.")


units_sold_summary(transactions_data)
print("\n")
mean_median_comparison(transactions_data)

Statistical summary of units sold


count    1825.000000
mean       22.332603
std        10.818861
min         4.000000
25%        14.000000
50%        21.000000
75%        29.000000
max        72.000000
Name: units_sold, dtype: float64



The mean value (22.33) is higher than the median (21.0) by 1.33.
This indicates a right-skewed distribution pulled toward high-sales days.


### Summary of Units Sold & Distribution Analysis

* **Mean:** 22.33
* **Median (50%):** 21.00
* **Standard Deviation:** 10.82
* **Minimum:** 4.00
* **Maximum:** 72.00

#### Key Insights & Comparison
* **Mean vs. Median:** The mean (22.33) is higher than the median (21.00) by **1.33 units**. 
* **Distribution Shape:** Because the mean is higher than the median, the distribution is right-skewed (pulled toward higher values). While the median represents the true physical middle of the dataset, the mean is being pulled upward by occasional high-sales days.
* **Standard Deviation:** A standard deviation of 10.82 indicates that daily sales typically fluctuate by about 11 units above or below the average.

#### Typical Daily Sales Volume
While this business averages about 22 units sold per day, a typical day actually sits closer to 21 units because a few exceptionally high-sales days (peaking at 72) are pulling the average upward.

<img src="distribution_chart_a.png" width="500" alt="Right Skewed Distribution">

In [6]:
round(transactions_data['units_sold'].skew(), 2)

np.float64(0.86)

In [7]:
transactions_data['units_sold'].kurtosis()

np.float64(0.6985572279824215)

In [8]:
def shape_distribution(df):
    """Figuring out the Shapeness and symmetery of the distribution"""

    skewness = df['units_sold'].skew()
    kurtosis = df['units_sold'].kurtosis()

    print(f"The skewness ({skewness:.2f}) is between 0.5 and 1.0 which indicates that distribution is moderately skewed.")
    print(f"The kurtosis ({kurtosis:.2f}) is above 0 which indicates the distribution peak is sharper than normal.")

shape_distribution(transactions_data)

The skewness (0.86) is between 0.5 and 1.0 which indicates that distribution is moderately skewed.
The kurtosis (0.70) is above 0 which indicates the distribution peak is sharper than normal.


## Summary of Shapeness of the Distribution

* **Skewness:** 0.86 (Moderately right-skewed)
* **Kurtosis:** 0.70 (Leptokurtic / Sharper peak than normal)

### Key Comparisons and Insights
* **Skewness Categories:** A skewness of 0.86 falls between 0.5 and 1.0, confirming that the distribution of daily sales is **moderately right-skewed**. This mathematically proves that the mean is being pulled upward by high-sales days.
* **Kurtosis Categories:** A kurtosis of 0.70 is above 0, indicating that the distribution has a **sharper peak than a normal distribution**. This means daily sales values are heavily concentrated around a central volume rather than being evenly spread out.

### Overall Shape Description
The overall shape shows that most sales days cluster heavily around low-to-mid volumes (typically between 14 and 29 units) with occasional, unexpected high spikes reaching up to 72 units.

In [9]:
def units_sold_ranges(df, bins, values):
    """Grouping the units sold data into group and show how many items fall on each group"""

    grouped_data = pd.cut(df['units_sold'], bins=bins, labels=values)
    return grouped_data.value_counts().sort_index().reset_index()

bins = [0, 10, 20, 30, 40, 50, 60, 70, 80]
values = ['0-10', '10-20', '20-30', '30-40', '40-50', '50-60', '60-70', '70-80']

units_sold_ranges(transactions_data, bins, values)

,units_sold,count
0,0-10,219
1,10-20,687
2,20-30,543
3,30-40,254
4,40-50,90
5,50-60,27
6,60-70,4
7,70-80,1


## Summary of units_sold into ranges

* **Most common bucket:** `10-20` units sold (687 days)
* **Least common bucket:** `70-80` units sold (1 day)

### Key Comparisons and Insights
* **Most Frequent Range:** The `10-20` bucket contains the most rows by a significant margin, meaning that on any given day, the business is most likely to clear between 10 and 20 sales. 
* **Rarity of High Volume:** The `70-80` bucket contains the fewest rows (just 1 day). Combined with the neighboring high-end buckets (`50-60` with 27 days and `60-70` with 4 days), this shows that genuinely high-volume sales days are **extremely rare, exceptional events** for this business, accounting for less than 2% of total operational days. Demand does not spread evenly across the range; instead, it is heavily concentrated at the lower end.
  

In [10]:
transactions_data.groupby('product_id')['units_sold'].describe()

,count,mean,std,min,25%,50%,75%,max
product_id,,,,,,,,
P001,365.0,26.295890,8.447973,7.0,20.0,25.0,31.0,57.0
P002,365.0,20.865753,6.747491,5.0,16.0,20.0,25.0,45.0
P003,365.0,35.112329,9.866749,14.0,29.0,34.0,41.0,72.0
P004,365.0,17.816438,5.925986,6.0,14.0,17.0,22.0,41.0
P005,365.0,11.572603,4.351116,4.0,9.0,11.0,14.0,25.0


In [14]:
def products_summary(df):
    """Splitting up the products and joing up with units sold to find out the statistical summary of each product"""

    return df.groupby('product_id')['units_sold'].describe()

def mean_std_max_units_sold(df):
    """Evaluating product mean, std, and maximum"""
    summary = products_summary(df)
    return summary[['mean', 'std', 'max']]

display(products_summary(transactions_data))    
display(mean_std_max_units_sold(transactions_data))

,count,mean,std,min,25%,50%,75%,max
product_id,,,,,,,,
P001,365.0,26.295890,8.447973,7.0,20.0,25.0,31.0,57.0
P002,365.0,20.865753,6.747491,5.0,16.0,20.0,25.0,45.0
P003,365.0,35.112329,9.866749,14.0,29.0,34.0,41.0,72.0
P004,365.0,17.816438,5.925986,6.0,14.0,17.0,22.0,41.0
P005,365.0,11.572603,4.351116,4.0,9.0,11.0,14.0,25.0


,mean,std,max
product_id,,,
P001,26.295890,8.447973,57.0
P002,20.865753,6.747491,45.0
P003,35.112329,9.866749,72.0
P004,17.816438,5.925986,41.0
P005,11.572603,4.351116,25.0


## Summary of Sales Performance by Product

### 1. Key Metrics Table
* **P001:** Mean = 26.30, Std Dev = 8.45, Max = 57.0
* **P002:** Mean = 20.87, Std Dev = 6.75, Max = 45.0
* **P003:** Mean = 35.11, Std Dev = 9.87, Max = 72.0
* **P004:** Mean = 17.82, Std Dev = 5.93, Max = 41.0
* **P005:** Mean = 11.57, Std Dev = 4.35, Max = 25.0

### 2. Insights & Comparisons
* **Volume Leaders:** **P003** generates the highest average volume with **35.11 units/day**, while **P005** has the lowest performance, averaging just **11.57 units/day**.
* **Volatility:** **P003** has the highest standard deviation (**9.87**), making its daily performance the most unpredictable and prone to extreme spikes (reaching a maximum of 72 units).

### 3. Impact on Stock Planning
Because **P003** has the most unpredictable daily sales, stock planners must maintain a larger safety stock buffer for this item compared to more stable products (like P005) to prevent costly stockouts during sudden demand spikes.

In [21]:
def product_skewness(df):
    """Calculating the skewness of each product and giving them the rank"""

    skew_df = df.groupby('product_id')['units_sold'].skew().reset_index()
    skew_df['rank'] = skew_df['units_sold'].rank(ascending=False, method='min')
    return skew_df

product_skewness(transactions_data)

,product_id,units_sold,rank
0,P001,0.742487,1.0
1,P002,0.553516,5.0
2,P003,0.559690,4.0
3,P004,0.561764,3.0
4,P005,0.653188,2.0


## Summary of Product Skewness & Modeling Implications

### 1. Skewness Rankings (Most to Least Skewed)
All five products exhibit positive skewness, ranking as follows:
1. **P001:** 0.742 (Rank 1 — Most Skewed)
2. **P005:** 0.653 (Rank 2)
3. **P004:** 0.562 (Rank 3)
4. **P003:** 0.559 (Rank 4)
5. **P002:** 0.554 (Rank 5 — Least Skewed)

### 2. Key Insights & Machine Learning Implications
* **Distribution Check:** All products fall between 0.5 and 1.0, meaning they are all **moderately right-skewed**. This confirms that every single product experiences occasional, abnormally high-volume sales days that pull its mean upward.
* **Handling Spikes in ML Models:** A predictive model trained on **P001** (or any of these products) **cannot safely ignore these spikes**. If ignored, the model will fail to predict high-demand events, leading to severe under-forecasting; therefore, the model must account for them by using engineering features (such as holiday/promotional flags) or utilizing outlier-resistant algorithms.

In [29]:
def count_outliers_per_product(df):
    """Calculating the outliers per product"""

    q1 = df.groupby('product_id')['units_sold'].transform(lambda x: np.percentile(x, 25))
    q3 = df.groupby('product_id')['units_sold'].transform(lambda x: np.percentile(x, 75))

    iqr = q3 - q1
    lower_fence = q1 - (1.5 * iqr)
    upper_fence = q3 + (1.5 * iqr)

    is_outlier = (df['units_sold'] < lower_fence) | (df['units_sold'] > upper_fence)

    outlier_counts = df[is_outlier].groupby('product_id')['units_sold'].size().reset_index(name='outlier count')

    return outlier_counts

count_outliers_per_product(transactions_data)

,product_id,outlier count
0,P001,6
1,P002,6
2,P003,7
3,P004,2
4,P005,9


## Summary of Outlier Analysis Using IQR Fences

### 1. Outlier Counts by Product
Using the standard Tukey's fence method ($IQR = Q3 - Q1$), the total number of high-sales outlier days for each product is:
* **P005:** 9 outlier days (Highest)
* **P003:** 7 outlier days
* **P001:** 6 outlier days
* **P002:** 6 outlier days
* **P004:** 2 outlier days (Lowest)

### 2. Key Insights & Real-World Interpretation
* **Most Outlier-Prone Product:** **P005** produces the highest number of outlier days (9 days). Given that P005 has the lowest baseline average sales volume (~11.5 units/day) across the catalog, these sudden spikes stand out drastically against its quiet day-to-day baseline.
* **Real-World Meaning:** In a business context, these extreme outlier days represent sudden, non-typical demand shocks. They likely correspond to **highly successful promotional campaigns**, **holiday shopping rushes**, or **bulk B2B/corporate orders** where a single customer buys far more inventory than a normal retail consumer.